In [0]:
%python
import json
kafka_connection_json=dbutils.secrets.get(scope="finguard-scope",key="kafka_connection_details")
kafka_config=json.loads(kafka_connection_json)
bootstrap_servers=kafka_config['bootstrap_servers']
api_key=kafka_config['api_key']
api_secret=kafka_config['api_secret']
topic=kafka_config['topic']


In [0]:
Select * from finguard.gold.transaciton_count_by_minute

In [0]:
select count(*) from finguard.bronze.transactions

In [0]:
%python
jaas_config=f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'

In [0]:
%python
streaming_batch = spark.read.format ("kafka")\
    .option("kafka.bootstrap.servers", bootstrap_servers)\
    .option("subscribe", topic)\
    .option("kafka.security.protocol", "SASL_SSL")\
    .option("kafka.sasl.mechanism", "PLAIN")\
    .option("kafka.sasl.jaas.config", jaas_config)\
    .option("startingOffsets", "earliest").load()
    

In [0]:
%python
streaming_batch.count()

In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.types import *

parsed_batch=streaming_batch.select(
col("key").cast("string"),
col("value").cast("string"),
col("topic"),
col("partition"),
col("offset"),
col("timestamp"),
col("timestampType")
)

In [0]:
%python
display(parsed_batch)

In [0]:
%python
parsed_batch.write.saveAsTable("finguard.bronze.transcation_batch_test")

In [0]:
%python
streaming_stream = spark.readStream.format ("kafka")\
    .option("kafka.bootstrap.servers", bootstrap_servers)\
    .option("subscribe", topic)\
    .option("kafka.security.protocol", "SASL_SSL")\
    .option("kafka.sasl.mechanism", "PLAIN")\
    .option("kafka.sasl.jaas.config", jaas_config)\
    .option("startingOffsets", "earliest").load()

In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.types import *

parsed_stream=streaming_stream.select(
col("key").cast("string"),
col("value").cast("string"),
col("topic"),
col("partition"),
col("offset"),
col("timestamp"),
col("timestampType")
)

In [0]:
%python
parsed_stream.writeStream.format("delta").outputMode("append").option("checkpointLocation", "/Volumes/finguard/source/checkpoint")\
                         .trigger(availableNow=True).toTable("finguard.bronze.transactions_streaming_test")